In [ ]:
!pip install bertopic datasets openai datamapplot

# 零、总述

本章主要探讨生成模型和表示模型如何在**无监督学习**领域发挥作用，尽管有监督算法比较盛行，但**无监督的方法由于能够在无需提前标注的情况下基于语义内容对文本进行分组，仍然具有巨大潜力**

下面要介绍的**文本聚类（Text Clustering）** 和 **主题建模(Topic Modeling)** 就是属于无监督学习算法

*文本聚类* 旨在基于文本的语义内容、含义和关系对相似文本进行分组

<img src="./resources/text_clustering.png">


*主题建模* 旨在找出这些文本都在讨论哪些主题，所以它关注的问题是：一个文本集合中，隐藏着哪些主题？每个文本又包含哪些主题？

<img src="./resources/topic_modeling.png">



首先，我们先探索如何使用嵌入模型进行聚类，然后过渡到一种受文本聚类启发的主题建模方法，即 BERTopic。

在本章中，我们将在 ArXiv 文章上运行聚类和主题建模算法。ArXiv 是一个主要面向计算机科学、数学和物理领域的开放的学术文章平台。下面我们将探索计算与语言领域的文章，对应于 arxiv_nlp 这一数据集，它包含 1991 年至 2024 年间来自 ArXiv cs.CL 板块的 44949 篇摘要。

下面我们加载数据，做数据准备

In [ ]:
from datasets import load_dataset
dataset = load_dataset("maartengr/arxiv_nlp")["train"]

abstracts = dataset["Abstracts"]
titles = dataset["Titles"]

# 一、文本聚类的通用流程

文本聚类不仅可以发现已知的数据模式，更可以挖掘不为人知的数据模式，它可以帮助你直观地理解任务及其复杂性。虽然文本聚类的方法有很多，从基于图的神经网络到基于质心的聚类技术，但当前比较流行的通用流程主要包含以下三个步骤（Embedding -> Reduction -> Cluster）：
1. 使用**嵌入模型（Embedding Model）** 将输入文档转换为嵌入向量
2. 使用**降维模型（Dimensionality Reduction Model）** 将嵌入向量降至更低维度空间
3. 使用**聚类模型（Cluster Model）** 对降维后的嵌入向量进行聚类

## 1.1 嵌入文档

这里我们选择 thenlper/gte-small 模型来作为嵌入模型

In [ ]:
from sentence_transformers import SentenceTransformer

# 为每个摘要创建嵌入向量
embedding_model = SentenceTransformer("thenlper/gte-small")
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

In [ ]:
embeddings.shape

我们可以看到嵌入向量的维度为 384 维，即每个摘要的语义表示（向量表示）包含了 384 个值

## 1.2 嵌入向量降维

降维：高维空间 -> 低维空间

**降维** 会带来两个问题：第一个是计算量会变得很大，但最主要的还是第二个：**高维空间会使数据变得非常“稀疏”**，也就是说，**维度越高，点之间距离越远**，其原理基于欧式距离的公式，公式如下：
$$
D = \sqrt{(a_1 - b_1)^2 + (a_2 - b_2)^2 + \cdot + (a_d - b_d)^2}
$$

因此，**距离会随着维度的平方根增长**，并且会带来一个很重要的问题：**距离的区分能力下降**，也就是*最近的点和最远的点之间，相对差异越来越小*，例如低维空间：
```text
最近邻：0.2
最远点：3.0
```

但是到达了高维空间之后，它会变成：
```text
最近邻：400
最远点：430
```

所以，**降维**不仅是为了减少计算量，更重要的是把数据从过于稀疏的高维空间压缩到一个更紧凑、更有意义的低维表示空间中。还有一点需要补充的是：**维度不是越低越好**，那样会丢失大量的有用信息，所以要在信息保留与降维之间做权衡。

我们介绍两个降维的方法：主成分分析（Principal Component Analysis,**PCA**）和统一流行逼近和投影(Uniform Manifold Approximation and Projection, **UMAP**)。

1. PCA: 主成分分析

它是一种经典的**线性降维算法**。PCA 将数据投影到一组新的正交坐标轴上，这些轴按照能够解释的数据方差从大到小排列。它核心的思想是**找到数据变化最大的方向，然后把数据投影到这些方向上**

2. UMAP：统一流行逼近和投影

它是一种**非线性降维算法/流形学习算法**，它更关注“谁和谁是邻居”，它的核心目标之一是**让低维空间尽量保持高维数据中的邻域结构**

本次流程我们使用的是 UMAP，因为它在处理非线性关系和结构方面比 PCA 表现更好

In [ ]:
from umap import UMAP

# 将嵌入向量从 384 维降至 5 维
umap_model = UMAP(
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

reduced_embeddings = umap_model.fit_transform(embeddings)

## 1.3 对降维后的嵌入向量进行聚类

聚类可以用于各种应用程序，包括：
- **客户细分**：你可以根据客户的购买记录和他们在网站上的活动对客户进行聚类
- **数据分析**
- **特征工程**
- **异常检测**：也叫离群值检测
- **半监督学习**
- **搜索引擎**：一些搜索引擎可让你搜索参考图像相似的图像
- **图像分割**：根据像素的颜色对像素进行聚类，然后用其聚类的平均颜色替换每个像素的颜色
- 等等。。。

关于聚类没有统一的定义，它实际上取决于上下文，并且不同的算法会得到不同种类的集群。一些算法会寻找围绕特定点（称为质心）的实例。其他算法则寻找密集实例连续区域：这些集群可以呈现任何形状

下面，我们来介绍几个聚类算法

1. K-Means 算法

KMeans 算法通过将样本分成 n 个相同方差的组来对数据进行聚类，它是会最小化一个被称为**惯性**（inertia）的指标或**簇内平方和**，公式如下：

$$
\sum_{i=0}^n \min_{\mu_j \in C}(||x_i - \mu_j||^2)
$$

K-Means 算法需指定簇的数量，它会将样本集 $X$ 分成 $K$ 个互不连接的 $C$ 个簇，每个簇都被簇中样本的平均值 $u_j$ 描述。这通常被称为簇的质心（centroids）, 这些质心并不是 $X$ 的样本点，即使它们处在相同的空间中。

惯性（Inertia）可以被当作是衡量每个簇有多么连贯，多么紧凑的指标。但是它也有一些缺点：
- 惯性（Inertia） 假设簇是**凸的**并且是**各向同性的**，但现实中的数据集并不总是这样。对于一些细长型的簇，以及有不规则形状的流形簇，它表现的不是很好
- 惯性（Inertia）不是一个归一化的指标：我们仅仅是将这个值降到最优，也就是0。但是，在高维空间中，欧式距离往往会被放大（这就是所谓的**维度的诅咒(curse of dimensionality)**）。所以，在执行 KMeans 算法之前需进行一个；类似于 PCA 的降维方法，不仅可以解除维度的诅咒，而且还可以加速计算

这里讲两个概念：**凸的（凸集）** 和 **各向同性**
a. 凸的（凸集，convex set）

直观的理解为：*在一个集合内部任意选两个点，把这两个点用一条直线连起来，整条线段仍在此集合内部*，那么这个集合就是凸集

在数学上，如果对于任意两个点 $x_1, x_2 \in C$，以及任意 $0 \le \lambda \le 1$, 都有：
$$
\lambda x_1 + (1 - \lambda)x_2 \in C
$$
则我们说，集合 C 是凸集，

b. 各向同性（isotropic）

从中心往各个方向看，数据的扩散程度大致相同

2. DBSCAN 和 HDBSCAN

3. GMM（Gaussian Mixture Models）



这里，我们使用算法 HDBSCAN 来对上述嵌入向量进行聚类

In [ ]:
from hdbscan import HDBSCAN

# 拟合模型并提取簇
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric="euclidean",
    cluster_selection_method="eom"
).fit(reduced_embeddings)

clusters = hdbscan_model.labels_

print(f"生成簇的数量为：{len(set(clusters))}")
print(f"簇的形状为：{clusters.shape}")

## 1.4 检查生成的簇

我们以生成的第一个簇为例

In [ ]:
import numpy as np

# 打印簇 0 中的前三个文档
cluster = 0
for index in np.where(clusters==cluster)[0][:3]:
    print(abstracts[index][:100] + "... \n\n")

从打印的文档来看，这个簇似乎主要包含有关手语翻译的文档

接下来，我们更进一步，可视化聚类的结果，我们将384维的向量降至二维以便于可视化（其结果仅仅是原始嵌入向量的一个近似表示）

In [ ]:
import pandas as pd

reduced_embeddings = UMAP(
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=42
).fit_transform(embeddings)

df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])
df["title"] = titles
df["cluster"] = [str(c) for c in clusters]

# 选择离群点和非离群点
clusters_points = df.loc[df.cluster != "-1", :]
outliers_points = df.loc[df.cluster == "-1", :]

In [ ]:
import matplotlib.pyplot as plt

# 分别绘制离群点和非离群点
plt.scatter(outliers_points.x, outliers_points.y, alpha=0.05, s=2, c="grey")
plt.scatter(
    clusters_points.x, clusters.y, c=clusters_points.cluster.astype(int),
    alpha=0.6, s=2, cmap="tab20b"
)
plt.axis("off")

# 二、从文本聚类到主题建模

**文本聚类** 是在大型文档集合中发现结构的有力工具，而**主题建模**的目标是找到一组最能代表、捕捉主题含义的关键词或词语

## 2.1 BERTopic: 一个模块化的主题建模框架

## 2.2 添加特殊的 “乐高积木块”

## 2.3 文本生成的 “乐高积木块”